# Walk-through of `hybrid`: the warm-up, with all of its code

Work package 2, first step: a neural network (no hybrid model yet) learns the pharmacokinetics and pharmacodynamics of a panel of virtual patients, which the mechanistic model simulates.

This notebook contains the code of the package `hybrid`, file by file, in the order in which a run uses it:

- A cell which starts with `%%source <file>` **is** that file of the repository. Running it writes the file into the clone of this session, if the cell differs from the file.
- The cells between the files run the step and show its result.
- To change the code, edit the cell and run it; the cells after it use the new code.
- The last section pushes your changes to a branch of your fork, for a pull request.

Run the cells from top to bottom. Generated files (dataset, weights, logs, figures) are written to Google Drive, not to the repository. The tests are not in this notebook; they are in `hybrid/tests/` of the clone (file browser on the left).

## Setup

The repository, the branch or commit, and the folder of Google Drive for the generated files.

In [ ]:
REPO = "https://github.com/matthiaskoenig/neural-ode.git"  # the repository, or your fork to push changes
REF = "main"  # branch, tag or commit
RUNS = "/content/drive/MyDrive/hybrid_runs"  # generated files; "MyDrive" is "Meine Ablage"

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/neural-ode"
if not os.path.isdir(REPO_DIR):  # a second run of this cell keeps your changes of the files
    !git clone -q {REPO} {REPO_DIR} && git -C {REPO_DIR} checkout -q {REF}
!pip install -q -e {REPO_DIR}/sbml2jax -e {REPO_DIR}/hybrid

In [ ]:
import hashlib
import logging
import os
import sys
import warnings
from pathlib import Path

from IPython.core.magic import register_cell_magic

os.makedirs(RUNS, exist_ok=True)
os.environ["HYBRID_RUNS"] = RUNS
sys.path[:0] = [f"{REPO_DIR}/hybrid/src", f"{REPO_DIR}/sbml2jax/src"]
logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)
# shell commands (lines with !) after JAX was started
warnings.filterwarnings("ignore", message=r"os\.fork\(\) was called")

# a changed file is imported again by the next cell which uses it
%load_ext autoreload
%autoreload 2

# sha256 of every file when this notebook was built
ORIGINAL = {
    "configs/warmup_apixaban.toml": "50f7e8b5a19ce4bbd98f88114670f519d2c6b86a587a0de990dddf25c9da714b",
    "src/hybrid/config.py": "49d243b22ac3dd8d99892847c9d2ecc29eb932352076894fefd08bd6a8485776",
    "src/hybrid/runs.py": "47a4a1c312c2d633701e2390f76587bdfdfdcc5532727b33bf9a11b3c8f667e0",
    "src/hybrid/data/population.py": "58261514de9a0c4c5d9e8e6e2db3e0625a3251f24c7022cee1259ce650ff2464",
    "src/hybrid/data/simulate.py": "28f40a0cea30eacb659f5edbbdb8918a4de34dff1fd717fe4d8b4cbb966b3d06",
    "src/hybrid/data/dataset.py": "0cf5add372f6286539ef71e26e43a029874815b896376805aa92e8c3a02bc38e",
    "src/hybrid/data/io.py": "c6309738e824885138a81aa9c6d4193592e8d18a47d77af49b4c0d0a4cfc7645",
    "src/hybrid/nn/mlp.py": "14ec03bc9016629339b8a76a5ce369cc19e23f59fc37a3f0b972dcb6cc4a8080",
    "src/hybrid/nn/scaling.py": "0fb02e3950a65d2501291c5ff9076f674124176db49d07314cae242e2235c6b9",
    "src/hybrid/nn/panel.py": "092ea68e708879722cb580f902b75f9bf4e2459670a354b00d88d4340b77f332",
    "src/hybrid/training/loss.py": "cb908bdb9371d4b05578a95a21bdbff066a7007bec8ddab2c712b652aea548b5",
    "src/hybrid/training/fit.py": "aef794860caa6fce3729c1c99024ebbead789d7b369dbeb9b3ecddabd7e7c185",
    "src/hybrid/training/checkpoint.py": "c2b7bc8c27b8d60c2dd49be68ff79c969306daf86ec44a73bd6773d504313ad8",
    "src/hybrid/evaluation/metrics.py": "8a5e694a2570dee767bf0f7405a20b86f53998b62959356251f7d3baa0fdaebd",
    "src/hybrid/evaluation/plots.py": "a49c3b2bb3f54e4cd7470c165ba055d6a88a638a9bce3f5460fbda45af5bb8e4",
    "src/hybrid/warmup.py": "4cf5057088a523aabf9e8a8cf27f2093edafa74473bb069a6b77953581b255d0",
    "src/hybrid/cli.py": "638d73630953ea3930c4965d8fc23d8e752e503331ab4bc96d8ed069d7185b64"
}


@register_cell_magic
def source(line, cell):
    """Writes the cell to a file of hybrid/ in the clone, if the cell differs from the file."""
    name = line.strip()
    path = Path(REPO_DIR) / "hybrid" / name
    text = cell.rstrip("\n") + "\n"
    on_disk = path.read_text() if path.exists() else None
    cell_is_original = hashlib.sha256(text.encode()).hexdigest() == ORIGINAL.get(name)
    if text == on_disk:
        print(f"{name}: unchanged")
    elif cell_is_original and on_disk is not None:
        # the cell was not edited, so the file is from another commit than this notebook
        print(f"{name}: NOT written. The file of the repository differs from this notebook;")
        print("open the notebook of the same branch or commit as REF.")
    else:
        path.write_text(text)
        print(f"{name}: written")


!git -C {REPO_DIR} log -1 --format="commit %h: %s"

## 1. A case is one file

Everything specific to a case is in a TOML file: the SBML model, how the patients differ (dose and covariates), what is measured (observables with their noise), the network and the training. The code below contains no model ids, doses or covariates; another substance is another file.

In [ ]:
%%source configs/warmup_apixaban.toml
# Warm-up of work package 2: a plain neural network (no hybrid model) learns the
# pharmacokinetics and pharmacodynamics of a panel of virtual patients, which is
# simulated with the mechanistic apixaban model.
#
# Everything specific to the case is in this file; a new substance is a new file.

name = "warmup_apixaban"
seed = 0

# the SBML model: pinned (url with {ref}, commit, checksum) or a local `path`
[model]
name = "apixaban"
url = "https://raw.githubusercontent.com/matthiaskoenig/apixaban-model/{ref}/models/apixaban_body_flat.xml"
ref = "d0e3277f3ebf389fe8ed8c9bc63be567ba929d3d"
sha256 = "eeba5a98213ee4b32c2c2320b61c8409a5f427d59761c406738c6e21e741c6bb"

[simulation]
# sampling times of every patient [min]: dense after the dose, sparse later
times = [0, 15, 30, 45, 60, 90, 120, 180, 240, 360, 480, 720, 960, 1200, 1440]
batch_size = 64  # patients simulated at once

# the dose is set at the first time; `target` is a state or constant of the model
[dose]
target = "PODOSE_api"  # oral dose [mg]
distribution = "choice"
values = [2.5, 5.0, 10.0, 20.0]

# covariates are inputs of the model which differ between the patients
[covariates.BW]  # body weight [kg]
distribution = "uniform"
low = 50.0
high = 110.0

[covariates.KI__f_renal_function]  # renal function, 1 is healthy
distribution = "loguniform"
low = 0.2
high = 1.2

[covariates.f_cirrhosis]  # degree of cirrhosis, 0 is healthy
distribution = "uniform"
low = 0.0
high = 0.6

# observables are outputs of the model, measured with the noise
# sigma = additive + proportional * value
[observables.Cve_api]  # venous plasma concentration [mmol/l]
proportional = 0.10
additive = 2e-6

[observables.antiXa_activity]  # anti-Xa activity [IU/ml]
proportional = 0.10
additive = 0.02

[observables.PT]  # prothrombin time [s]
proportional = 0.03
additive = 0.0

[observables.aPTT]  # activated partial thromboplastin time [s]
proportional = 0.03
additive = 0.0

[population]
n_subjects = 256
validation_fraction = 0.25  # of the patients, never seen in training
missing_fraction = 0.0  # of the measurements, dropped at random

[network]
width = 64
depth = 3
activation = "tanh"

[training]
batch_size = 32  # patients per Adam step
adam_steps = 4000
learning_rate = 3e-3
lbfgs_steps = 300
eval_every = 100

`config.py` reads and checks the file.

In [ ]:
%%source src/hybrid/config.py
"""The configuration of a case: one TOML file with everything specific to it.

The code contains no model ids, doses, units or covariates. A case names its
SBML model, how the patients differ (dose and covariates), what is measured
(observables with their noise), the network and the training.
"""

import hashlib
import itertools
import json
import tomllib
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from sbml2jax.pinned import PinnedModel, fetch

DISTRIBUTIONS = ("uniform", "loguniform", "choice")
ACTIVATIONS = ("tanh", "relu", "gelu", "softplus")


class ConfigError(ValueError):
    """A configuration file which cannot be used, with the reason."""


@dataclass(frozen=True)
class ModelConfig:
    """The SBML model: a local file or a pinned download.

    Attributes:
        name: Name of the model (`name` of the table, else the name of the case), the
            directory of a pinned model in the cache.
        path: Local SBML file, or `None` for a pinned model.
        url: URL of a pinned model with the placeholder `{ref}`.
        ref: Commit of a pinned model.
        sha256: Checksum of a pinned model.
    """

    name: str
    path: Path | None = None
    url: str | None = None
    ref: str | None = None
    sha256: str | None = None

    def sbml_path(self) -> Path:
        """The SBML file; a pinned model is downloaded into the cache and verified.

        Raises:
            ConfigError: The local file does not exist.
        """
        if self.path is not None:
            if not self.path.is_file():
                raise ConfigError(f"[model] path {self.path} does not exist")
            return self.path
        assert self.url is not None and self.ref is not None
        return fetch(PinnedModel(name=self.name, url=self.url, ref=self.ref, sha256=self.sha256))


@dataclass(frozen=True)
class Distribution:
    """How a quantity varies between the patients.

    Attributes:
        kind: `uniform` and `loguniform` between `low` and `high`, or `choice` of `values`.
        low: Lower bound.
        high: Upper bound.
        values: The values of a `choice`.
    """

    kind: str
    low: float = 0.0
    high: float = 0.0
    values: tuple[float, ...] = ()

    @property
    def log(self) -> bool:
        """Whether the network sees the logarithm of the quantity (positive, over a ratio scale)."""
        if self.kind == "loguniform":
            return True
        return self.kind == "choice" and min(self.values) > 0


@dataclass(frozen=True)
class Observable:
    """A measured output of the model with the noise `sigma = additive + proportional * value`."""

    proportional: float
    additive: float


@dataclass(frozen=True)
class NetworkConfig:
    """Hidden layers of the network."""

    width: int
    depth: int
    activation: str


@dataclass(frozen=True)
class TrainingConfig:
    """Settings of the training: Adam on batches of patients, then L-BFGS on all."""

    batch_size: int
    adam_steps: int
    learning_rate: float
    lbfgs_steps: int
    eval_every: int


@dataclass(frozen=True)
class Config:
    """A case, read from a TOML file with `read_config`.

    Attributes:
        name: Name of the case, the directory of its runs.
        seed: Seed of the population, the noise, the split and the training.
        model: The SBML model.
        times: Sampling times of every patient, in the time unit of the model.
        simulation_batch_size: Number of patients simulated at once.
        rtol: Relative tolerance of the simulation.
        atol: Absolute tolerance of the simulation.
        dose_target: State or constant which is set to the dose at the first time.
        dose: Distribution of the dose.
        covariates: Inputs of the model which differ between the patients.
        observables: Measured outputs of the model with their noise.
        n_subjects: Number of patients.
        validation_fraction: Fraction of the patients held out of the training.
        missing_fraction: Fraction of the measurements dropped at random.
        network: Hidden layers of the network.
        training: Settings of the training.
        text: The text of the file, copied into every run.
    """

    name: str
    seed: int
    model: ModelConfig
    times: tuple[float, ...]
    simulation_batch_size: int
    rtol: float
    atol: float
    dose_target: str
    dose: Distribution
    covariates: dict[str, Distribution]
    observables: dict[str, Observable]
    n_subjects: int
    validation_fraction: float
    missing_fraction: float
    network: NetworkConfig
    training: TrainingConfig
    text: str

    @property
    def subject_inputs(self) -> tuple[str, ...]:
        """Names of the quantities which differ between the patients: the dose target, then the covariates."""
        return (self.dose_target, *self.covariates)

    @property
    def subject_distributions(self) -> tuple[Distribution, ...]:
        """Distributions of `subject_inputs`."""
        return (self.dose, *self.covariates.values())

    def data_hash(self) -> str:
        """Identifies the dataset: a hash of everything the generated data depends on."""
        raw = tomllib.loads(self.text)
        keys = ("seed", "model", "simulation", "dose", "covariates", "observables", "population")
        content = json.dumps({key: raw.get(key) for key in keys}, sort_keys=True)
        return hashlib.sha256(content.encode()).hexdigest()[:10]


def _distribution(table: dict[str, Any], where: str) -> Distribution:
    kind = table.get("distribution")
    if kind not in DISTRIBUTIONS:
        raise ConfigError(f"[{where}] distribution must be one of {DISTRIBUTIONS}, not {kind!r}")
    if kind == "choice":
        values = tuple(float(value) for value in table.get("values", ()))
        if not values:
            raise ConfigError(f"[{where}] a choice needs `values`")
        return Distribution(kind=kind, values=values)
    try:
        low, high = float(table["low"]), float(table["high"])
    except KeyError as error:
        raise ConfigError(f"[{where}] {kind} needs `low` and `high`") from error
    if not low < high:
        raise ConfigError(f"[{where}] low must be below high")
    if kind == "loguniform" and low <= 0:
        raise ConfigError(f"[{where}] loguniform needs a positive `low`")
    return Distribution(kind=kind, low=low, high=high)


def _model(table: dict[str, Any], name: str, directory: Path) -> ModelConfig:
    name = str(table.get("name", name))
    if "path" in table:
        return ModelConfig(name=name, path=(directory / table["path"]).resolve())
    missing = [key for key in ("url", "ref", "sha256") if key not in table]
    if missing:
        raise ConfigError(
            f"[model] needs `path`, or `url`, `ref` and `sha256` (missing: {', '.join(missing)})"
        )
    return ModelConfig(name=name, url=table["url"], ref=table["ref"], sha256=table["sha256"])


def _fraction(value: float, where: str) -> float:
    if not 0.0 <= value < 1.0:
        raise ConfigError(f"{where} must be in [0, 1), not {value}")
    return float(value)


def read_config(path: Path) -> Config:
    """Reads and checks the TOML file of a case.

    Ids of the model (dose target, covariates, observables) are checked against
    the model when the data is generated, see `hybrid.data.simulate.panel_model`.

    Raises:
        ConfigError: A table or key is missing or has a value which cannot be used.
    """
    path = Path(path)
    text = path.read_text()
    raw = tomllib.loads(text)
    try:
        simulation, population = raw["simulation"], raw["population"]
        network, training = raw["network"], raw["training"]
        times = tuple(float(time) for time in simulation["times"])
        if len(times) < 2 or any(b <= a for a, b in itertools.pairwise(times)):
            raise ConfigError("[simulation] times must be at least two increasing values")
        observables = {
            name: Observable(proportional=float(table["proportional"]), additive=float(table["additive"]))
            for name, table in raw["observables"].items()
        }
        if not observables:
            raise ConfigError("[observables] needs at least one observable")
        if any(
            o.proportional < 0 or o.additive < 0 or o.proportional + o.additive == 0
            for o in observables.values()
        ):
            raise ConfigError("[observables] noise must be non-negative and not zero")
        if network["activation"] not in ACTIVATIONS:
            raise ConfigError(f"[network] activation must be one of {ACTIVATIONS}")
        config = Config(
            name=str(raw["name"]),
            seed=int(raw["seed"]),
            model=_model(raw["model"], str(raw["name"]), path.parent),
            times=times,
            simulation_batch_size=int(simulation.get("batch_size", 64)),
            rtol=float(simulation.get("rtol", 1e-8)),
            atol=float(simulation.get("atol", 1e-10)),
            dose_target=str(raw["dose"]["target"]),
            dose=_distribution(raw["dose"], "dose"),
            covariates={
                name: _distribution(table, f"covariates.{name}")
                for name, table in raw.get("covariates", {}).items()
            },
            observables=observables,
            n_subjects=int(population["n_subjects"]),
            validation_fraction=_fraction(
                population["validation_fraction"], "[population] validation_fraction"
            ),
            missing_fraction=_fraction(
                population.get("missing_fraction", 0.0), "[population] missing_fraction"
            ),
            network=NetworkConfig(
                width=int(network["width"]),
                depth=int(network["depth"]),
                activation=str(network["activation"]),
            ),
            training=TrainingConfig(
                batch_size=int(training["batch_size"]),
                adam_steps=int(training["adam_steps"]),
                learning_rate=float(training["learning_rate"]),
                lbfgs_steps=int(training["lbfgs_steps"]),
                eval_every=int(training["eval_every"]),
            ),
            text=text,
        )
    except KeyError as error:
        raise ConfigError(f"{path.name}: missing {error}") from error
    if config.n_subjects < 2 or config.simulation_batch_size < 1:
        raise ConfigError("[population] n_subjects must be at least 2 and [simulation] batch_size at least 1")
    if config.dose_target in config.covariates:
        raise ConfigError(f"{config.dose_target!r} is the dose target and cannot be a covariate")
    return config

In [ ]:
from hybrid.config import read_config

CASE = f"{REPO_DIR}/hybrid/configs/warmup_apixaban.toml"
config = read_config(CASE)
print("patients differ in:", config.subject_inputs)
print("observables:       ", tuple(config.observables))
print("sampling times:    ", config.times)
print("dataset id:        ", config.data_hash())

## 2. Where generated files go

Datasets, weights, logs and figures are not part of the repository. `runs.py` puts them below one directory, here the folder of Google Drive, and records for every run which commit, configuration and seed produced it.

In [ ]:
%%source src/hybrid/runs.py
"""Where generated files go: datasets, weights, logs and figures are not part of the repository.

All of them are written below one directory, `HYBRID_RUNS` or
`~/.cache/hybrid/runs`. In Colab the variable points to a folder of Google
Drive. Every run records the commit, the configuration and the seed which
produced it.
"""

import json
import os
import platform
import subprocess
import sys
from datetime import datetime
from importlib import metadata
from pathlib import Path

from hybrid.config import Config

#: environment variable with the directory of the generated files
RUNS_ENV = "HYBRID_RUNS"


def runs_dir() -> Path:
    """Root of the generated files, `HYBRID_RUNS` or `~/.cache/hybrid/runs`."""
    return Path(os.environ.get(RUNS_ENV, Path.home() / ".cache" / "hybrid" / "runs"))


def dataset_path(config: Config) -> Path:
    """File of the dataset of a case; a change of the data settings gives another file."""
    return runs_dir() / "datasets" / f"{config.name}-{config.data_hash()}" / "dataset.npz"


def commit() -> tuple[str, bool]:
    """The commit of the code and whether it has uncommitted changes.

    Read from the git repository of an editable installation, else from the
    installation record of pip (`pip install git+...`); `("unknown", False)`
    if neither exists.
    """
    directory = Path(__file__).parent
    try:
        head = subprocess.run(  # noqa: S603
            ["git", "-C", str(directory), "rev-parse", "HEAD"],  # noqa: S607
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
        status = subprocess.run(  # noqa: S603
            ["git", "-C", str(directory), "status", "--porcelain"],  # noqa: S607
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
        return head, bool(status)
    except (OSError, subprocess.CalledProcessError):
        pass
    try:
        direct_url = metadata.distribution("hybrid").read_text("direct_url.json")
        return json.loads(direct_url or "{}")["vcs_info"]["commit_id"], False
    except (metadata.PackageNotFoundError, KeyError, json.JSONDecodeError):
        return "unknown", False


def new_run(config: Config) -> Path:
    """Creates the directory of a new run with `config.toml` and `run.json`.

    The directory is `<runs>/<name>/<time>-<commit>`. `run.json` holds what
    produced the run: commit, seed, dataset, versions of the packages and the
    device.
    """
    import jax

    head, dirty = commit()
    started = datetime.now().astimezone()
    directory = runs_dir() / config.name / f"{started:%Y%m%d-%H%M%S}-{head[:7]}"
    directory.mkdir(parents=True, exist_ok=False)
    (directory / "config.toml").write_text(config.text)
    record = {
        "name": config.name,
        "started": started.isoformat(timespec="seconds"),
        "commit": head,
        "uncommitted_changes": dirty,
        "seed": config.seed,
        "dataset": str(dataset_path(config).relative_to(runs_dir())),
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "device": jax.default_backend(),
        "packages": {
            name: metadata.version(name)
            for name in ("hybrid", "sbml2jax", "jax", "equinox", "optax", "diffrax", "numpy")
        },
    }
    (directory / "run.json").write_text(json.dumps(record, indent=2) + "\n")
    return directory

In [ ]:
from hybrid.runs import commit, dataset_path, runs_dir

print("generated files:", runs_dir())
print("dataset of the case:", dataset_path(config).relative_to(runs_dir()))
print("commit of the code, uncommitted changes:", commit())

## 3. Data generation

### Virtual patients

The dose and the covariates of every patient are sampled from the distributions of the case.

In [ ]:
%%source src/hybrid/data/population.py
"""Virtual patients: the dose and the covariates of every patient, sampled from the configuration."""

import numpy as np

from hybrid.config import Config, Distribution


def sample(distribution: Distribution, n: int, rng: np.random.Generator) -> np.ndarray:
    """`n` values of a distribution."""
    if distribution.kind == "choice":
        return rng.choice(np.asarray(distribution.values), size=n)
    if distribution.kind == "loguniform":
        return np.exp(rng.uniform(np.log(distribution.low), np.log(distribution.high), size=n))
    return rng.uniform(distribution.low, distribution.high, size=n)


def sample_population(config: Config, rng: np.random.Generator) -> np.ndarray:
    """The patients of a case.

    Returns:
        Array of shape `(n_subjects, len(config.subject_inputs))`: the dose and
        the covariates of every patient, in the order of `config.subject_inputs`.
    """
    columns = [sample(distribution, config.n_subjects, rng) for distribution in config.subject_distributions]
    return np.stack(columns, axis=1)

In [ ]:
import numpy as np
import pandas as pd

from hybrid.data.population import sample_population

x = sample_population(config, np.random.default_rng(0))
patients = pd.DataFrame(x, columns=config.subject_inputs)
display(patients.head())
patients.describe().loc[["min", "mean", "max"]]

### Simulation and noise

All patients are simulated with the mechanistic model: one function for one patient, batched over the patients with `jax.vmap` and compiled with `jax.jit`. Then normal noise with `sigma = additive + proportional * value` is added per observable.

In [ ]:
%%source src/hybrid/data/simulate.py
"""Simulation of the panel with the mechanistic model, and the measurement noise."""

from collections.abc import Callable

import jax
import jax.numpy as jnp
import numpy as np
from sbml2jax.model import JaxModel, Segment, SolverSettings

from hybrid.config import Config, ConfigError


def panel_model(config: Config) -> JaxModel:
    """The mechanistic model of a case, with its ids checked against the configuration.

    Raises:
        ConfigError: A covariate is not an input, an observable is not an output,
            or the dose target is not a state or constant of the model.
    """
    model = JaxModel.from_sbml(config.model.sbml_path())
    module = model.module
    unknown = [name for name in config.covariates if name not in module.INPUTS]
    if unknown:
        raise ConfigError(f"covariates {unknown} are not inputs of the model")
    unknown = [name for name in config.observables if name not in module.OUTPUTS]
    if unknown:
        raise ConfigError(f"observables {unknown} are not outputs of the model")
    if config.dose_target not in (*module.STATES, *module.CONSTANTS):
        raise ConfigError(f"dose target {config.dose_target!r} is not a state or constant of the model")
    return model


def subject_simulator(model: JaxModel, config: Config) -> Callable[[jax.Array], jax.Array]:
    """The simulation of one patient as a function of its row of the population.

    The function maps the dose and the covariates `(len(config.subject_inputs),)`
    to the observables at the sampling times `(n_times, n_observables)`. It can be
    traced, so it is batched with `jax.vmap` and compiled with `jax.jit`.
    """
    module = model.module
    ts = np.asarray(config.times)
    covariates = jnp.asarray([module.INPUTS.index(name) for name in config.covariates], dtype=int)
    observables = np.asarray([module.OUTPUTS.index(name) for name in config.observables])
    settings = SolverSettings(rtol=config.rtol, atol=config.atol)

    def simulate(x: jax.Array) -> jax.Array:
        p = model.p.at[covariates].set(x[1:]) if covariates.size else model.p
        subject = JaxModel(p=p, module=module)
        segments = [Segment(end=float(ts[-1]), changes={config.dose_target: x[0]})]
        return subject.simulate(ts, segments, settings)[:, observables]

    return simulate


def simulate_panel(model: JaxModel, config: Config, x: np.ndarray) -> np.ndarray:
    """The observables of every patient without noise.

    The patients are simulated in batches of `config.simulation_batch_size` with
    one compiled function; the last batch is filled up, so it is compiled once.

    Returns:
        Array of shape `(n_subjects, n_times, n_observables)`.
    """
    batch = jax.jit(jax.vmap(subject_simulator(model, config)))
    size = min(config.simulation_batch_size, len(x))
    ys = []
    for start in range(0, len(x), size):
        rows = x[start : start + size]
        filled = np.concatenate([rows, np.repeat(rows[-1:], size - len(rows), axis=0)])
        ys.append(np.asarray(batch(jnp.asarray(filled)))[: len(rows)])
    y = np.concatenate(ys)
    if not np.all(np.isfinite(y)):
        raise FloatingPointError("the simulation of the panel gave values which are not finite")
    return y


def add_noise(y_clean: np.ndarray, config: Config, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    """Measurements of the simulated observables.

    Normal noise with `sigma = additive + proportional * value` per observable;
    `config.missing_fraction` of the measurements are dropped at random.

    Returns:
        The measurements and the mask of the measurements which exist, both of
        the shape of `y_clean`.
    """
    proportional = np.asarray([o.proportional for o in config.observables.values()])
    additive = np.asarray([o.additive for o in config.observables.values()])
    sigma = additive + proportional * np.abs(y_clean)
    y = y_clean + sigma * rng.standard_normal(y_clean.shape)
    mask = rng.uniform(size=y_clean.shape) >= config.missing_fraction
    return y, mask

In [ ]:
import matplotlib.pyplot as plt

from hybrid.data.simulate import add_noise, panel_model, simulate_panel

model = panel_model(config)  # downloads the pinned model once, checks the ids of the case
y_clean = simulate_panel(model, config, x[:8])  # the first 8 patients; compiles first
y, mask = add_noise(y_clean, config, np.random.default_rng(1))

figure, axes = plt.subplots(1, len(config.observables), figsize=(4 * len(config.observables), 3))
for k, (axis, name) in enumerate(zip(np.atleast_1d(axes), config.observables)):
    axis.plot(config.times, y_clean[:, :, k].T, linewidth=1)
    axis.scatter(config.times, y[0, :, k], s=12, color="black", zorder=3)
    axis.set_title(name)
    axis.set_xlabel("time")
figure.suptitle("8 patients without noise (lines), measurements of the first patient (dots)")
figure.tight_layout()

### The panel

`Panel` holds the measurements of all patients and the split by patient: a patient is either used for training or for validation, never both. `generate_panel` runs the three steps above for all patients; `io.py` saves the panel as one file.

In [ ]:
%%source src/hybrid/data/dataset.py
"""The panel: many patients with one time course each, split by patient."""

from dataclasses import dataclass, replace

import numpy as np

from hybrid.config import Config
from hybrid.data.population import sample_population
from hybrid.data.simulate import add_noise, panel_model, simulate_panel

#: streams of the random numbers of a case, so that one does not shift another
STREAM_POPULATION, STREAM_NOISE, STREAM_SPLIT, STREAM_TRAINING = range(4)


def rng(config: Config, stream: int) -> np.random.Generator:
    """The random number generator of one stream of a case."""
    return np.random.default_rng([config.seed, stream])


@dataclass(frozen=True)
class Panel:
    """Measurements of a panel of patients at common sampling times.

    Attributes:
        times: Sampling times `(n_times,)`.
        x: Dose and covariates of the patients `(n_subjects, n_inputs)`.
        y: Measurements `(n_subjects, n_times, n_observables)`.
        y_clean: The simulated values without noise, of the shape of `y`.
        mask: Which measurements exist, of the shape of `y`.
        train: Which patients are used for training `(n_subjects,)`; the others validate.
        x_names: Names of the columns of `x`.
        y_names: Names of the observables.
        log_x: Which columns of `x` the network sees as logarithm.
    """

    times: np.ndarray
    x: np.ndarray
    y: np.ndarray
    y_clean: np.ndarray
    mask: np.ndarray
    train: np.ndarray
    x_names: tuple[str, ...]
    y_names: tuple[str, ...]
    log_x: tuple[bool, ...]

    def subset(self, train: bool) -> "Panel":
        """The training patients, or the validation patients."""
        rows = self.train if train else ~self.train
        return replace(
            self,
            x=self.x[rows],
            y=self.y[rows],
            y_clean=self.y_clean[rows],
            mask=self.mask[rows],
            train=self.train[rows],
        )


def split_subjects(n: int, validation_fraction: float, generator: np.random.Generator) -> np.ndarray:
    """A random split by patient: `True` for training.

    At least one patient is in each part, so no patient contributes
    measurements to both.
    """
    n_validation = min(max(round(n * validation_fraction), 1), n - 1)
    train = np.ones(n, dtype=bool)
    train[generator.permutation(n)[:n_validation]] = False
    return train


def generate_panel(config: Config) -> Panel:
    """Samples the patients of a case, simulates them and adds the noise."""
    x = sample_population(config, rng(config, STREAM_POPULATION))
    y_clean = simulate_panel(panel_model(config), config, x)
    y, mask = add_noise(y_clean, config, rng(config, STREAM_NOISE))
    return Panel(
        times=np.asarray(config.times),
        x=x,
        y=y,
        y_clean=y_clean,
        mask=mask,
        train=split_subjects(config.n_subjects, config.validation_fraction, rng(config, STREAM_SPLIT)),
        x_names=config.subject_inputs,
        y_names=tuple(config.observables),
        log_x=tuple(distribution.log for distribution in config.subject_distributions),
    )

In [ ]:
%%source src/hybrid/data/io.py
"""Saving and loading a panel as one `.npz` file."""

from dataclasses import asdict
from pathlib import Path

import numpy as np

from hybrid.data.dataset import Panel

_TUPLES = ("x_names", "y_names", "log_x")


def save_panel(path: Path, panel: Panel) -> None:
    """Writes the panel to `path`."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as file:
        np.savez_compressed(file, **{key: np.asarray(value) for key, value in asdict(panel).items()})


def load_panel(path: Path) -> Panel:
    """Reads a panel written by `save_panel`."""
    with np.load(path, allow_pickle=False) as file:
        fields = {key: file[key] for key in file.files}
    for key in _TUPLES:
        fields[key] = tuple(fields[key].tolist())
    return Panel(**fields)

In [ ]:
from hybrid.data.dataset import generate_panel
from hybrid.data.io import load_panel, save_panel

path = dataset_path(config)
if path.exists():
    panel = load_panel(path)
else:
    panel = generate_panel(config)  # all patients, about half a minute
    save_panel(path, panel)
print(path)
print("measurements (patients, times, observables):", panel.y.shape)
print("training patients:", panel.train.sum(), " validation patients:", (~panel.train).sum())

## 4. The network

`mlp.py` builds a plain multilayer perceptron. `zero_last` is not used in the warm-up; the hybrid models need it to start as the mechanistic model.

In [ ]:
%%source src/hybrid/nn/mlp.py
"""A multilayer perceptron with several inputs and outputs."""

from collections.abc import Callable

import equinox as eqx
import jax
import jax.numpy as jnp

ACTIVATIONS: dict[str, Callable[[jax.Array], jax.Array]] = {
    "tanh": jnp.tanh,
    "relu": jax.nn.relu,
    "gelu": jax.nn.gelu,
    "softplus": jax.nn.softplus,
}


def mlp(
    n_in: int, n_out: int, width: int, depth: int, activation: str, key: jax.Array, zero_last: bool = False
) -> eqx.nn.MLP:
    """A network of `depth` hidden layers of `width` units, vector to vector.

    Args:
        n_in: Number of inputs.
        n_out: Number of outputs.
        width: Units of a hidden layer.
        depth: Number of hidden layers.
        activation: Name of the activation, a key of `ACTIVATIONS`.
        key: Random key of the initial weights.
        zero_last: Sets the weights and biases of the last layer to zero, so
            the network gives zero for every input at initialization. A hybrid
            model with such a network starts as the mechanistic model.
    """
    network = eqx.nn.MLP(n_in, n_out, width, depth, activation=ACTIVATIONS[activation], key=key)
    if zero_last:
        last = network.layers[-1]
        network = eqx.tree_at(
            lambda n: (n.layers[-1].weight, n.layers[-1].bias),
            network,
            (jnp.zeros_like(last.weight), jnp.zeros_like(last.bias)),
        )
    return network

`scaling.py` standardizes the inputs and outputs. The constants are fitted to the training patients, saved with the weights and not trained.

In [ ]:
%%source src/hybrid/nn/scaling.py
"""Scaling of the inputs and outputs of a network, as part of the network.

The constants are fitted to the training data once and saved with the weights,
so a loaded network needs no data. They are numpy arrays, which the training
leaves unchanged (it only updates JAX arrays).
"""

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np


class Scaler(eqx.Module):
    """Standardizes a vector, some columns after a logarithm.

    Attributes:
        log: Which columns are transformed with the logarithm first.
        shift: Mean of the (transformed) training values.
        scale: Standard deviation of the (transformed) training values.
    """

    log: np.ndarray
    shift: np.ndarray
    scale: np.ndarray

    @classmethod
    def fit(cls, values: np.ndarray, log: tuple[bool, ...] | None = None) -> "Scaler":
        """The scaler of training values `(n, n_columns)`; a constant column keeps scale 1."""
        log_columns = np.zeros(values.shape[1], dtype=bool) if log is None else np.asarray(log, dtype=bool)
        if np.any(values[:, log_columns] <= 0):
            raise ValueError("a column with logarithm has values which are not positive")
        transformed = np.where(log_columns, np.log(np.where(log_columns, values, 1.0)), values)
        scale = transformed.std(axis=0)
        return cls(log=log_columns, shift=transformed.mean(axis=0), scale=np.where(scale > 0, scale, 1.0))

    def __call__(self, values: jax.Array) -> jax.Array:
        """Scales a vector `(n_columns,)`."""
        transformed = jnp.where(self.log, jnp.log(jnp.where(self.log, values, 1.0)), values)
        return (transformed - self.shift) / self.scale

    def inverse(self, scaled: jax.Array) -> jax.Array:
        """The values of a scaled vector."""
        transformed = scaled * self.scale + self.shift
        return jnp.where(self.log, jnp.exp(jnp.where(self.log, transformed, 0.0)), transformed)

`panel.py` is the network of the warm-up: it maps a time and the inputs of a patient (dose and covariates) to all observables.

In [ ]:
%%source src/hybrid/nn/panel.py
"""The network of the warm-up: time and patient to the observables."""

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np

from hybrid.config import NetworkConfig
from hybrid.data.dataset import Panel
from hybrid.nn.mlp import mlp
from hybrid.nn.scaling import Scaler


class PanelNetwork(eqx.Module):
    """Maps a time and the inputs of a patient to all observables.

    The time enters twice, linear and as `log1p(time / time_step)`, because a
    time course changes fast after the dose and slowly later.

    Attributes:
        time_step: The smallest sampling interval, the unit of the logarithmic time.
        scale_in: Scaling of the two time features and the inputs of the patient.
        network: The multilayer perceptron on the scaled features.
        scale_out: Scaling of the observables; the network predicts scaled values.
    """

    time_step: float
    scale_in: Scaler
    network: eqx.nn.MLP
    scale_out: Scaler

    @classmethod
    def create(cls, panel: Panel, config: NetworkConfig, key: jax.Array) -> "PanelNetwork":
        """A network with random weights, scaled to the training patients of the panel."""
        train = panel.subset(train=True)
        time_step = float(np.min(np.diff(panel.times)))
        features = np.concatenate(
            [
                np.tile(_time_features(panel.times, time_step), (len(train.x), 1)),
                np.repeat(train.x, len(panel.times), axis=0),
            ],
            axis=1,
        )
        n_out = len(panel.y_names)
        return cls(
            time_step=time_step,
            scale_in=Scaler.fit(features, (False, False, *panel.log_x)),
            network=mlp(features.shape[1], n_out, config.width, config.depth, config.activation, key),
            scale_out=_output_scaler(train),
        )

    def __call__(self, time: jax.Array, x: jax.Array) -> jax.Array:
        """The observables `(n_observables,)` of a patient with the inputs `x` at one time."""
        features = jnp.concatenate([jnp.stack([time, jnp.log1p(time / self.time_step)]), x])
        return self.scale_out.inverse(self.network(self.scale_in(features)))

    def predict(self, times: jax.Array, x: jax.Array) -> jax.Array:
        """The time courses `(n_subjects, n_times, n_observables)` of patients `(n_subjects, n_inputs)`."""
        return jax.vmap(lambda row: jax.vmap(lambda time: self(time, row))(times))(x)


def _time_features(times: np.ndarray, time_step: float) -> np.ndarray:
    return np.stack([times, np.log1p(times / time_step)], axis=1)


def _output_scaler(train: Panel) -> Scaler:
    """Mean and standard deviation of the measurements which exist, per observable."""
    values = np.where(train.mask, train.y, np.nan).reshape(-1, len(train.y_names))
    scale = np.nanstd(values, axis=0)
    return Scaler(
        log=np.zeros(values.shape[1], dtype=bool),
        shift=np.nanmean(values, axis=0),
        scale=np.where(scale > 0, scale, 1.0),
    )

In [ ]:
import equinox as eqx
import jax
import jax.numpy as jnp

from hybrid.nn.panel import PanelNetwork
from hybrid.training.fit import is_trainable

network = PanelNetwork.create(panel, config.network, jax.random.PRNGKey(config.seed))
weights = jax.tree_util.tree_leaves(eqx.filter(network, is_trainable))
print("trained numbers:", sum(leaf.size for leaf in weights))

times = jnp.asarray(panel.times)
prediction = network.predict(times, jnp.asarray(panel.x))
print("prediction (patients, times, observables):", prediction.shape)
print("one patient at one time, untrained:", dict(zip(panel.y_names, np.asarray(prediction[0, 5]).round(4).tolist())))

## 5. Training

### The loss

The negative log-likelihood of the measurements under a normal error model. The size of the noise is not given to the training: the additive and the proportional part of every observable are trained with the network.

In [ ]:
%%source src/hybrid/training/loss.py
"""The loss: negative log-likelihood of the measurements under a normal error model."""

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np


def gaussian_nll(
    prediction: jax.Array, observation: jax.Array, sigma: jax.Array, mask: jax.Array
) -> jax.Array:
    """Mean negative log-likelihood of the measurements which exist.

    Args:
        prediction: Predicted values.
        observation: Measurements, of the shape of `prediction`; values without
            a measurement are ignored and may be anything finite.
        sigma: Standard deviation of every measurement, of the same shape.
        mask: Which measurements exist, of the same shape.
    """
    residual = jnp.where(mask, (observation - prediction) / sigma, 0.0)
    terms = jnp.log(sigma) + 0.5 * residual**2 + 0.5 * jnp.log(2 * jnp.pi)
    return jnp.sum(jnp.where(mask, terms, 0.0)) / jnp.sum(mask)


def _inverse_softplus(value: np.ndarray) -> np.ndarray:
    return np.log(np.expm1(value))


class ErrorModel(eqx.Module):
    """The combined error model `sigma = additive + proportional * |prediction|` per observable.

    Both parts are trained with the network, as positive numbers through a
    softplus. The additive part is in units of `scale`, the scale of each
    observable, so the same initial value fits every observable.

    Attributes:
        raw_additive: Additive part before the softplus, in units of `scale`.
        raw_proportional: Proportional part before the softplus.
        scale: Scale of every observable (not trained).
    """

    raw_additive: jax.Array
    raw_proportional: jax.Array
    scale: np.ndarray

    @classmethod
    def create(cls, scale: np.ndarray, additive: float = 1.0, proportional: float = 0.05) -> "ErrorModel":
        """An error model which starts as a constant sigma of about the scale of the observable.

        The loss is then close to a mean squared error of the scaled values at
        first, which keeps the start of the training stable.
        """
        ones = np.ones_like(np.asarray(scale, dtype=float))
        return cls(
            raw_additive=jnp.asarray(_inverse_softplus(additive * ones)),
            raw_proportional=jnp.asarray(_inverse_softplus(proportional * ones)),
            scale=np.asarray(scale, dtype=float),
        )

    @property
    def additive(self) -> jax.Array:
        """Additive part in the units of the observables."""
        return self.scale * jax.nn.softplus(self.raw_additive)

    @property
    def proportional(self) -> jax.Array:
        """Proportional part."""
        return jax.nn.softplus(self.raw_proportional)

    def sigma(self, prediction: jax.Array) -> jax.Array:
        """Standard deviation of the measurements of predictions `(..., n_observables)`."""
        return self.additive + self.proportional * jnp.abs(prediction)

In [ ]:
from hybrid.training.loss import ErrorModel, gaussian_nll

error = ErrorModel.create(network.scale_out.scale)
print("initial sigma of every observable at its mean:", np.asarray(error.sigma(jnp.asarray(network.scale_out.shift))))
initial = gaussian_nll(prediction, jnp.asarray(panel.y), error.sigma(prediction), jnp.asarray(panel.mask))
print("loss of the untrained network:", float(initial))

### The optimization

Adam on batches of patients, then L-BFGS on all training patients. The model with the lowest loss on the validation patients is returned. `fit` knows nothing about the warm-up; the hybrid models train with the same function.

In [ ]:
%%source src/hybrid/training/fit.py
"""The optimization: Adam on batches of patients, then L-BFGS on all training patients.

`fit` knows nothing about the model: it trains the JAX arrays of any pytree
(an equinox module, or a tuple of modules) on a loss of the model and the data
of a batch of patients.
"""

import logging
from collections.abc import Callable
from dataclasses import dataclass
from typing import Any, TypeVar

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import optax

logger = logging.getLogger(__name__)

Model = TypeVar("Model")
#: loss of a model on the data of some patients: `loss(model, *arrays) -> scalar`
Loss = Callable[..., jax.Array]


@dataclass(frozen=True)
class FitSettings:
    """Settings of `fit`.

    Attributes:
        batch_size: Patients per Adam step.
        adam_steps: Number of Adam steps; the learning rate decays to 1 % on a cosine.
        learning_rate: Initial learning rate of Adam.
        lbfgs_steps: Number of L-BFGS steps on all training patients, after Adam.
        eval_every: Adam steps between two evaluations on all patients.
        clip: Maximal global norm of an Adam gradient.
    """

    batch_size: int
    adam_steps: int
    learning_rate: float
    lbfgs_steps: int
    eval_every: int
    clip: float = 1.0


def is_trainable(leaf: Any) -> bool:
    """Whether a leaf of a model is trained: floating point JAX arrays, not numpy constants."""
    return isinstance(leaf, jax.Array) and jnp.issubdtype(leaf.dtype, jnp.inexact)


def fit(
    model: Model,
    loss: Loss,
    train: tuple[np.ndarray, ...],
    valid: tuple[np.ndarray, ...],
    settings: FitSettings,
    rng: np.random.Generator,
) -> tuple[Model, list[dict[str, float | int | str]]]:
    """Trains a model and returns it at its lowest validation loss.

    Args:
        model: Pytree whose floating point JAX arrays are trained.
        loss: Loss of the model on the arrays of some patients.
        train: Arrays of the training patients, the patients on the first axis.
        valid: Arrays of the validation patients, only used to choose the returned model.
        settings: Settings of the optimization.
        rng: Random numbers of the batches.

    Returns:
        The model with the lowest validation loss of all evaluations, and the
        log: one entry per evaluation with `step`, `phase` (`adam` or `lbfgs`),
        `train` and `valid` loss.
    """
    params, static = eqx.partition(model, is_trainable)
    train_arrays = tuple(jnp.asarray(array) for array in train)
    valid_arrays = tuple(jnp.asarray(array) for array in valid)

    def objective(p: Model, *arrays: jax.Array) -> jax.Array:
        return loss(eqx.combine(p, static), *arrays)

    evaluate = jax.jit(objective)
    log: list[dict[str, float | int | str]] = []
    best = (np.inf, params)

    def record(step: int, phase: str, p: Model) -> None:
        nonlocal best
        entry = {
            "step": step,
            "phase": phase,
            "train": float(evaluate(p, *train_arrays)),
            "valid": float(evaluate(p, *valid_arrays)),
        }
        if not np.isfinite(entry["train"]):
            raise FloatingPointError(f"the training loss is not finite at step {step} ({phase})")
        log.append(entry)
        logger.info("%-5s step %5d  train %.4f  valid %.4f", phase, step, entry["train"], entry["valid"])
        if entry["valid"] < best[0]:
            best = (entry["valid"], p)

    # Adam on batches of patients
    schedule = optax.cosine_decay_schedule(settings.learning_rate, max(settings.adam_steps, 1), alpha=0.01)
    adam = optax.chain(optax.clip_by_global_norm(settings.clip), optax.adam(schedule))

    @jax.jit
    def adam_step(p: Model, state: optax.OptState, *arrays: jax.Array) -> tuple[Model, optax.OptState]:
        updates, state = adam.update(jax.grad(objective)(p, *arrays), state, p)
        return optax.apply_updates(p, updates), state

    n = len(train_arrays[0])
    size = min(settings.batch_size, n)
    state = adam.init(params)
    record(0, "adam", params)
    order = rng.permutation(n)
    for step in range(1, settings.adam_steps + 1):
        if len(order) < size:
            order = rng.permutation(n)
        rows, order = order[:size], order[size:]
        params, state = adam_step(params, state, *(array[rows] for array in train_arrays))
        if step % settings.eval_every == 0 or step == settings.adam_steps:
            record(step, "adam", params)

    # L-BFGS on all training patients, from the best model of Adam
    if settings.lbfgs_steps > 0:
        params = best[1]
        lbfgs = optax.lbfgs()

        def full(p: Model) -> jax.Array:
            return objective(p, *train_arrays)

        value_and_grad = optax.value_and_grad_from_state(full)

        @jax.jit
        def lbfgs_step(p: Model, state: optax.OptState) -> tuple[Model, optax.OptState]:
            value, grad = value_and_grad(p, state=state)
            updates, state = lbfgs.update(grad, state, p, value=value, grad=grad, value_fn=full)
            return optax.apply_updates(p, updates), state

        state = lbfgs.init(params)
        for step in range(1, settings.lbfgs_steps + 1):
            params, state = lbfgs_step(params, state)
            if step % max(settings.eval_every // 10, 1) == 0 or step == settings.lbfgs_steps:
                record(settings.adam_steps + step, "lbfgs", params)

    return eqx.combine(best[1], static), log

In [ ]:
%%source src/hybrid/training/checkpoint.py
"""Saving and loading the arrays of a model."""

from pathlib import Path
from typing import TypeVar

import equinox as eqx

Model = TypeVar("Model")


def save_weights(path: Path, model: Model) -> None:
    """Writes all arrays of the model (weights and scaling constants) to `path`."""
    path.parent.mkdir(parents=True, exist_ok=True)
    eqx.tree_serialise_leaves(path, model)


def load_weights(path: Path, like: Model) -> Model:
    """Reads the arrays written by `save_weights` into a model of the same structure as `like`."""
    return eqx.tree_deserialise_leaves(path, like)

## 6. Evaluation

Errors per observable, and the figures of a run.

In [ ]:
%%source src/hybrid/evaluation/metrics.py
"""Errors of predictions per observable."""

import numpy as np

from hybrid.data.dataset import Panel


def panel_metrics(prediction: np.ndarray, panel: Panel) -> dict[str, dict[str, float]]:
    """Errors of the predictions of a panel, per observable.

    Args:
        prediction: Predictions of the shape of `panel.y`.
        panel: The patients which were predicted.

    Returns:
        For every observable: `rmse_data` against the measurements which exist,
        `rmse_truth` against the simulated values without noise, `nrmse_truth`
        (`rmse_truth` relative to the standard deviation of the simulated
        values) and `r2_truth`.
    """
    metrics = {}
    for k, name in enumerate(panel.y_names):
        mask = panel.mask[..., k]
        truth = panel.y_clean[..., k]
        error_truth = prediction[..., k] - truth
        rmse_truth = float(np.sqrt(np.mean(error_truth**2)))
        variance = float(np.var(truth))
        metrics[name] = {
            "rmse_data": float(np.sqrt(np.mean((prediction[..., k] - panel.y[..., k])[mask] ** 2))),
            "rmse_truth": rmse_truth,
            "nrmse_truth": rmse_truth / np.sqrt(variance) if variance > 0 else float("nan"),
            "r2_truth": 1.0 - float(np.mean(error_truth**2)) / variance if variance > 0 else float("nan"),
        }
    return metrics

In [ ]:
%%source src/hybrid/evaluation/plots.py
"""Figures of a trained model: fits of single patients, predicted against simulated, learning curves."""

from pathlib import Path

import numpy as np
from matplotlib.figure import Figure

from hybrid.data.dataset import Panel

NETWORK = "#2a78d6"
MECHANISTIC = "#eb6834"
INK = "#3d3d3a"
MUTED = "#8a8980"
GRID = "#e4e3dc"


def _style(axis) -> None:
    axis.spines[["top", "right"]].set_visible(False)
    axis.spines[["left", "bottom"]].set_color(MUTED)
    axis.tick_params(colors=MUTED, labelsize=7, length=3)
    axis.grid(color=GRID, linewidth=0.6)
    axis.set_axisbelow(True)


def _save(figure: Figure, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(path, dpi=200, facecolor="white")


def spread_subjects(panel: Panel, n: int) -> np.ndarray:
    """Rows of `n` patients spread over the first input (the dose), then the second."""
    order = np.lexsort(panel.x.T[::-1])
    return order[np.linspace(0, len(order) - 1, min(n, len(order))).round().astype(int)]


def plot_fits(
    path: Path,
    panel: Panel,
    rows: np.ndarray,
    dense_times: np.ndarray,
    dense_prediction: np.ndarray,
    title: str,
) -> None:
    """Time courses of single patients: measurements, mechanistic model and network.

    Args:
        path: File of the figure.
        panel: The patients, usually the validation patients.
        rows: The patients which are shown, one column each.
        dense_times: Times of the predictions of the network.
        dense_prediction: Predictions `(len(rows), len(dense_times), n_observables)`.
        title: Title of the figure.
    """
    n_rows, n_columns = len(panel.y_names), len(rows)
    figure = Figure(figsize=(2.4 * n_columns + 0.6, 1.7 * n_rows + 1.0), layout="constrained")
    axes = figure.subplots(n_rows, n_columns, sharex=True, sharey="row", squeeze=False)
    for j, row in enumerate(rows):
        inputs = ", ".join(
            f"{name} {value:.3g}" for name, value in zip(panel.x_names, panel.x[row], strict=True)
        )
        axes[0, j].set_title(inputs.replace(", ", "\n"), fontsize=7, color=INK, loc="left")
        for k in range(n_rows):
            axis = axes[k, j]
            _style(axis)
            mask = panel.mask[row, :, k]
            (truth,) = axis.plot(
                panel.times, panel.y_clean[row, :, k], color=MECHANISTIC, linewidth=1.5, linestyle="--"
            )
            (network,) = axis.plot(dense_times, dense_prediction[j, :, k], color=NETWORK, linewidth=1.8)
            data = axis.scatter(
                panel.times[mask],
                panel.y[row, mask, k],
                s=12,
                color=INK,
                edgecolors="white",
                linewidths=0.5,
                zorder=3,
            )
    for k, name in enumerate(panel.y_names):
        axes[k, 0].set_ylabel(name, fontsize=8, color=INK)
    for axis in axes[-1]:
        axis.set_xlabel("time", fontsize=8, color=INK)
    figure.suptitle(title, fontsize=10, color=INK, x=0.01, ha="left")
    figure.legend(
        [data, truth, network],
        ["measurements", "mechanistic model", "network"],
        loc="outside lower center",
        ncols=3,
        frameon=False,
        fontsize=8,
        labelcolor=INK,
    )
    _save(figure, path)


def plot_parity(path: Path, panel: Panel, prediction: np.ndarray, title: str) -> None:
    """Predictions of the network against the simulated values without noise, one panel per observable."""
    n = len(panel.y_names)
    figure = Figure(figsize=(2.6 * n + 0.4, 3.1), layout="constrained")
    axes = figure.subplots(1, n, squeeze=False)[0]
    for k, (axis, name) in enumerate(zip(axes, panel.y_names, strict=True)):
        _style(axis)
        truth, predicted = panel.y_clean[..., k].ravel(), prediction[..., k].ravel()
        low, high = min(truth.min(), predicted.min()), max(truth.max(), predicted.max())
        axis.plot([low, high], [low, high], color=MUTED, linewidth=1.0)
        axis.scatter(truth, predicted, s=6, color=NETWORK, alpha=0.5, linewidths=0)
        axis.set_title(name, fontsize=8, color=INK, loc="left")
        axis.set_xlabel("mechanistic model", fontsize=8, color=INK)
        axis.set_aspect("equal", adjustable="datalim")
    axes[0].set_ylabel("network", fontsize=8, color=INK)
    figure.suptitle(title, fontsize=10, color=INK, x=0.01, ha="left")
    _save(figure, path)


def plot_learning(path: Path, log: list[dict], title: str) -> None:
    """Training and validation loss over the steps; a line marks the start of L-BFGS."""
    figure = Figure(figsize=(5.2, 3.2), layout="constrained")
    axis = figure.subplots()
    _style(axis)
    steps = np.asarray([entry["step"] for entry in log])
    axis.plot(
        steps, [entry["train"] for entry in log], color=NETWORK, linewidth=1.8, label="training patients"
    )
    axis.plot(
        steps,
        [entry["valid"] for entry in log],
        color=MECHANISTIC,
        linewidth=1.8,
        label="validation patients",
    )
    lbfgs = [entry["step"] for entry in log if entry["phase"] == "lbfgs"]
    if lbfgs:
        start = max(entry["step"] for entry in log if entry["phase"] == "adam")
        axis.axvline(start, color=MUTED, linewidth=1.0, linestyle=":")
        axis.annotate(
            " L-BFGS", (start, 1.0), xycoords=("data", "axes fraction"), va="top", fontsize=7, color=MUTED
        )
    axis.set_xlabel("step", fontsize=8, color=INK)
    axis.set_ylabel("negative log-likelihood per measurement", fontsize=8, color=INK)
    axis.legend(frameon=False, fontsize=8, labelcolor=INK)
    axis.set_title(title, fontsize=10, color=INK, loc="left")
    _save(figure, path)

## 7. The warm-up: generate, train, evaluate

`warmup.py` puts the steps together, and `cli.py` is the command `hybrid` which calls it (`hybrid train <case>` in a terminal).

In [ ]:
%%source src/hybrid/warmup.py
"""The warm-up: a plain network learns the panel which the mechanistic model simulates.

No hybrid model yet. The steps are the ones of every later variant: generate
the data, train, evaluate; each writes its files below `hybrid.runs.runs_dir`.
"""

import csv
import json
import logging
from pathlib import Path

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np

from hybrid.config import Config, read_config
from hybrid.data.dataset import STREAM_TRAINING, Panel, generate_panel, rng
from hybrid.data.io import load_panel, save_panel
from hybrid.evaluation.metrics import panel_metrics
from hybrid.evaluation.plots import plot_fits, plot_learning, plot_parity, spread_subjects
from hybrid.nn.panel import PanelNetwork
from hybrid.runs import dataset_path, new_run, runs_dir
from hybrid.training.checkpoint import load_weights, save_weights
from hybrid.training.fit import FitSettings, fit
from hybrid.training.loss import ErrorModel, gaussian_nll

logger = logging.getLogger(__name__)


class WarmupModel(eqx.Module):
    """What is trained in the warm-up: the network and the error model of the measurements."""

    network: PanelNetwork
    error: ErrorModel


def dataset(config: Config, regenerate: bool = False) -> Panel:
    """The panel of a case: loaded from its file, or generated and saved if it does not exist."""
    path = dataset_path(config)
    if path.is_file() and not regenerate:
        logger.info("dataset %s", path)
        return load_panel(path)
    logger.info("generating %d patients", config.n_subjects)
    panel = generate_panel(config)
    save_panel(path, panel)
    logger.info("dataset written to %s", path)
    return panel


def build(config: Config, panel: Panel) -> WarmupModel:
    """The untrained model of a case, with the initial weights of its seed."""
    network = PanelNetwork.create(panel, config.network, jax.random.PRNGKey(config.seed))
    return WarmupModel(network=network, error=ErrorModel.create(network.scale_out.scale))


def loss(model: WarmupModel, times: jax.Array, x: jax.Array, y: jax.Array, mask: jax.Array) -> jax.Array:
    """Negative log-likelihood of the measurements `y` of the patients `x`."""
    prediction = model.network.predict(times, x)
    return gaussian_nll(prediction, y, model.error.sigma(prediction), mask)


def train(config: Config, regenerate: bool = False) -> Path:
    """Trains the network of a case and evaluates it.

    Returns:
        The directory of the run with `config.toml`, `run.json`, `weights.eqx`,
        `log.csv`, `metrics.json` and the figures.
    """
    panel = dataset(config, regenerate)
    directory = new_run(config)
    logger.info("run %s", directory)
    times = jnp.asarray(panel.times)
    parts = [panel.subset(train=True), panel.subset(train=False)]
    settings = FitSettings(
        batch_size=config.training.batch_size,
        adam_steps=config.training.adam_steps,
        learning_rate=config.training.learning_rate,
        lbfgs_steps=config.training.lbfgs_steps,
        eval_every=config.training.eval_every,
    )
    model, log = fit(
        build(config, panel),
        lambda m, x, y, mask: loss(m, times, x, y, mask),
        *((part.x, part.y, part.mask) for part in parts),
        settings,
        rng(config, STREAM_TRAINING),
    )
    save_weights(directory / "weights.eqx", model)
    with (directory / "log.csv").open("w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=["step", "phase", "train", "valid"])
        writer.writeheader()
        writer.writerows(log)
    evaluate(directory)
    return directory


def load(directory: Path) -> tuple[Config, Panel, WarmupModel]:
    """The configuration, the panel and the trained model of a run."""
    config = read_config(directory / "config.toml")
    record = json.loads((directory / "run.json").read_text())
    panel = load_panel(runs_dir() / record["dataset"])
    return config, panel, load_weights(directory / "weights.eqx", build(config, panel))


def evaluate(directory: Path) -> dict:
    """Writes `metrics.json` and the figures of a run from its saved weights.

    Returns:
        The metrics: errors per observable on the training and the validation
        patients, and the trained error model next to the one of the data.
    """
    config, panel, model = load(directory)
    predict = jax.jit(lambda times, x: model.network.predict(times, x))
    times = jnp.asarray(panel.times)
    metrics: dict = {"observables": {}, "error_model": {}}
    for name, part in (("train", panel.subset(train=True)), ("valid", panel.subset(train=False))):
        prediction = np.asarray(predict(times, jnp.asarray(part.x)))
        metrics["observables"][name] = panel_metrics(prediction, part)
        metrics[f"n_{name}"] = len(part.x)
    for k, (name, observable) in enumerate(config.observables.items()):
        metrics["error_model"][name] = {
            "proportional": float(model.error.proportional[k]),
            "proportional_data": observable.proportional,
            "additive": float(model.error.additive[k]),
            "additive_data": observable.additive,
        }
    (directory / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")

    valid = panel.subset(train=False)
    rows = spread_subjects(valid, 4)
    dense = np.linspace(panel.times[0], panel.times[-1], 201)
    plot_fits(
        directory / "fits.png",
        valid,
        rows,
        dense,
        np.asarray(predict(jnp.asarray(dense), jnp.asarray(valid.x[rows]))),
        f"{config.name}: validation patients",
    )
    plot_parity(
        directory / "parity.png",
        valid,
        np.asarray(predict(times, jnp.asarray(valid.x))),
        f"{config.name}: validation patients, all sampling times",
    )
    log_path = directory / "log.csv"
    if log_path.is_file():
        with log_path.open(newline="") as file:
            log = [
                {
                    "step": int(row["step"]),
                    "phase": row["phase"],
                    "train": float(row["train"]),
                    "valid": float(row["valid"]),
                }
                for row in csv.DictReader(file)
            ]
        plot_learning(directory / "learning.png", log, f"{config.name}: training")
    return metrics

In [ ]:
%%source src/hybrid/cli.py
"""Command line: `hybrid generate`, `hybrid train` and `hybrid evaluate`."""

import argparse
import json
import logging
from pathlib import Path

from hybrid import warmup
from hybrid.config import read_config
from hybrid.runs import RUNS_ENV, runs_dir


def main(argv: list[str] | None = None) -> None:
    """Entry point of the command `hybrid`."""
    parser = argparse.ArgumentParser(
        prog="hybrid",
        description=f"Neural networks on PK/PD data. Generated files go to ${RUNS_ENV} (now: {runs_dir()}).",
    )
    commands = parser.add_subparsers(dest="command", required=True)
    generate = commands.add_parser("generate", help="simulate the panel of a case and save it")
    generate.add_argument("config", type=Path, help="TOML file of the case")
    train = commands.add_parser("train", help="train the network of a case and evaluate it")
    train.add_argument("config", type=Path, help="TOML file of the case")
    train.add_argument(
        "--regenerate", action="store_true", help="simulate the panel again even if its file exists"
    )
    evaluate = commands.add_parser("evaluate", help="write the metrics and figures of a run again")
    evaluate.add_argument("run", type=Path, help="directory of the run")
    args = parser.parse_args(argv)

    logging.basicConfig(level=logging.INFO, format="%(message)s")
    if args.command == "generate":
        warmup.dataset(read_config(args.config), regenerate=True)
    elif args.command == "train":
        directory = warmup.train(read_config(args.config), regenerate=args.regenerate)
        print((directory / "metrics.json").read_text())
    else:
        print(json.dumps(warmup.evaluate(args.run), indent=2))


if __name__ == "__main__":
    main()

In [ ]:
from hybrid import warmup

config = read_config(CASE)  # read again, in case the case was changed above
run = warmup.train(config)

### Results

The run on Google Drive: `config.toml`, `run.json` (commit, seed, versions), `weights.eqx`, `log.csv`, `metrics.json` and the figures. The errors are against the mechanistic model without noise, on the validation patients, which the training has not seen.

In [ ]:
import json

from IPython.display import Image

metrics = json.loads((run / "metrics.json").read_text())
print(run)
display(pd.DataFrame(metrics["observables"]["valid"]).T)
display(pd.DataFrame(metrics["error_model"]).T)
for name in ("fits.png", "parity.png", "learning.png"):
    display(Image(filename=str(run / name), width=900))

## 8. Changing the code

Edit a cell above and run it, then run the cells which use it. Examples:

- another network: `width`, `depth` or `activation` in the cell of `configs/warmup_apixaban.toml`, then the training cell of section 7
- more patients or other covariates: the same cell; a new dataset is generated because the data settings changed
- another loss or optimizer: the cells of `training/loss.py` or `training/fit.py`

`git diff` shows what differs from the commit of this session.

In [ ]:
!git -C {REPO_DIR} status --short && git -C {REPO_DIR} diff --stat

### Checks

The checks of the repository: lint, format and the tests (a few minutes). They must pass before a pull request is merged.

In [ ]:
!pip install -q ruff pytest pytest-xdist
!cd {REPO_DIR}/hybrid && ruff check && ruff format --check --diff && python -m pytest -q -x -n 2

### Push the changes

Commits the changed files to a new branch and pushes it. This needs `REPO` to be a repository you may push to (your fork) and a Colab secret `GITHUB_TOKEN_CLASSIC` with a token for it. Nothing is pushed to `main`; open a pull request from the branch.

In [ ]:
BRANCH = "my-change"  # a new branch of your fork, not main
MESSAGE = "hybrid: describe the change"
GIT_NAME = ""  # author of the commit
GIT_EMAIL = ""

import subprocess

from google.colab import userdata

assert BRANCH not in ("main", "develop"), "push to a branch of its own and open a pull request"
assert GIT_NAME and GIT_EMAIL, "set the author of the commit"
token = userdata.get("GITHUB_TOKEN_CLASSIC")  # Colab secret with a token which may push to REPO


def git(*args):
    result = subprocess.run(["git", "-C", REPO_DIR, *args], capture_output=True, text=True)
    print((result.stdout + result.stderr).replace(token, "***").strip())
    result.check_returncode()


# the notebook is generated from the files, so it is built again from your changes
!cd {REPO_DIR}/hybrid && python notebooks/build_walkthrough.py
git("checkout", "-B", BRANCH)
git("add", "-A")
git("status", "--short")
git("-c", f"user.name={GIT_NAME}", "-c", f"user.email={GIT_EMAIL}", "commit", "-m", MESSAGE)
git("push", REPO.replace("https://", f"https://{token}@"), f"HEAD:refs/heads/{BRANCH}")
print(f"open the pull request: {REPO.removesuffix('.git')}/compare/{BRANCH}?expand=1")